## S&P 500 TR Index data quality verification

### Approach :

- Main dataset : load the **Timeseries** sheet of `CLEAN_RECOLTE DONNEES-EXCEL.xlsx`
- Load equivalent dataset from yahoo-finance and from financedatabase
- Compare monthly returns
- Save the aligned timeseries to `cache/cache-sp500-tr-indices.json`

### What the data represents

Each row is one month of the S&P 500 total return index.

- **`Period`** — month label (`Jan 2010`, `Feb 2010`, …)
- **`Date`** — the **exact trading day** the Lipper level was taken from (resolved against Yahoo’s daily calendar)
- **`Time`** — which session price is stored (`Open` / `High` / `Low` / `Close`); here always **`Close`**

Lipper’s rule for `Date`:
- if the 1st of the month is a trading day → that day’s close
- otherwise usually the **first trading day of the month**
- on **22** January/September holiday month-starts → the **previous session’s close** instead (e.g. Period `Jan 2010` → Date `2009-12-31`)

Yahoo and FinanceDatabase prices are taken on the **same `Date`** so the three columns are comparable.

- **S&P 500 TR (Lipper)** — Excel workbook (Lipper / Refinitiv)
- **S&P 500 TR (Yahoo)** — Yahoo Finance (`^SP500TR`)
- **S&P 500 TR (FinanceDatabase)** — FinanceDatabase Indices (`^SP500TR`), prices via FinanceToolkit (tries FMP first, falls back to YahooFinance if FMP still gates this ticker)

In [11]:
import sys
from pathlib import Path

import pandas as pd

FILENAME = "CLEAN_RECOLTE DONNEES-EXCEL.xlsx"
RELATIVE = Path("input-data-alix") / "clean" / FILENAME

def find_workbook() -> Path:
    """Find the clean workbook whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        candidate = directory / RELATIVE
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {RELATIVE}, starting from {start}"
    )

workbook = find_workbook()
code = workbook.parent / "funds"
if str(code) not in sys.path:
    sys.path.insert(0, str(code))

# Read the worksheet as before
timeseries_df = pd.read_excel(
    workbook,
    sheet_name="Timeseries",
    engine="openpyxl",
    dtype={"ISIN": "string", "RIC": "string"},
)

# Long series names live in the row where id == "name", not in the column headers.
# Headers are short codes ("S&P 500 TR", "S&P 500 Price"); dates live in column "id".
lipper_name = "S&P 500 COMPOSITE - TOT RETURN IND"
names = timeseries_df.loc[timeseries_df["id"].eq("name")].iloc[0].drop("id")
lipper_column = names[names.eq(lipper_name)].index[0]

meta_labels = {"name", "category", "fee (%/year)", "Note", "Date"}
sp500_indices = (
    timeseries_df.loc[~timeseries_df["id"].isin(meta_labels), ["id", lipper_column]]
    .rename(columns={"id": "month_start", lipper_column: "S&P 500 TR (Lipper)"})
    .assign(month_start=lambda frame: pd.to_datetime(frame["month_start"]))
)
sp500_indices["S&P 500 TR (Lipper)"] = pd.to_numeric(
    sp500_indices["S&P 500 TR (Lipper)"], errors="coerce"
)
sp500_indices["Period"] = sp500_indices["month_start"].dt.strftime("%b %Y")
sp500_indices = sp500_indices.reset_index(drop=True)

In [12]:
import yfinance as yf

# Resolve Lipper's exact observation Date against Yahoo daily closes, then take
# Yahoo's Close on that same Date. history() treats end as exclusive.
# Start a few days early so January rows can match the prior December session.
TOL = 0.02
start = sp500_indices["month_start"].min() - pd.Timedelta(days=10)
end = sp500_indices["month_start"].max() + pd.DateOffset(months=1)

daily = yf.Ticker("^SP500TR").history(
    start=start,
    end=end,
    interval="1d",
    auto_adjust=False,
    repair=False,
)
daily.index = daily.index.tz_localize(None)
closes = daily["Close"]

observation_dates = []
for month_start, lipper in zip(
    sp500_indices["month_start"], sp500_indices["S&P 500 TR (Lipper)"]
):
    month_closes = closes.loc[
        (closes.index >= month_start)
        & (closes.index < month_start + pd.DateOffset(months=1))
    ]
    first_date = month_closes.index[0]
    prior = closes.loc[closes.index < month_start]
    prior_date = prior.index[-1]

    if month_start in closes.index and abs(closes.loc[month_start] - lipper) <= TOL:
        observation_dates.append(month_start)
    elif abs(prior.iloc[-1] - lipper) <= TOL:
        observation_dates.append(prior_date)
    else:
        observation_dates.append(first_date)

sp500_indices["Date"] = observation_dates
sp500_indices["Time"] = "Close"
sp500_indices["S&P 500 TR (Yahoo)"] = sp500_indices["Date"].map(closes)

sp500_indices = sp500_indices[
    [
        "Period",
        "Date",
        "Time",
        "S&P 500 TR (Lipper)",
        "S&P 500 TR (Yahoo)",
    ]
]
sp500_indices

,Period,Date,Time,S&P 500 TR (Lipper),S&P 500 TR (Yahoo)
0,Jan 2010,2009-12-31,Close,1837.50,1837.500000
1,Feb 2010,2010-02-01,Close,1796.67,1796.670044
2,Mar 2010,2010-03-01,Close,1844.87,1844.869995
3,Apr 2010,2010-04-01,Close,1950.91,1950.910034
4,May 2010,2010-05-03,Close,1992.87,1992.869995
...,...,...,...,...,...
197,Jun 2026,2026-06-01,Close,16981.10,16981.099609
198,Jul 2026,2026-07-01,Close,16738.56,16738.560547
199,Aug 2026,2026-08-03,Close,17012.38,17012.380859
200,Sep 2026,2026-09-01,Close,17099.00,17099.000000


In [13]:
import os

import financedatabase as fd
from dotenv import find_dotenv, load_dotenv
from financedatabase.frame_model import FinanceFrame

# FinanceDatabase catalogues ^SP500TR ("S&P 500 (TR)"); prices come via FinanceToolkit.
# Prefer FMP; fall back to YahooFinance if this total-return index is still plan-gated.
# Map Close onto the same observation Date already resolved for Lipper/Yahoo.
load_dotenv(find_dotenv(usecwd=True))
api_key = os.getenv("FMP_API_KEY")
if not api_key:
    raise RuntimeError("FMP_API_KEY is missing from .env")

sp500_tr = FinanceFrame(fd.Indices().select().loc[["^SP500TR"]])
start = sp500_indices["Date"].min() - pd.Timedelta(days=5)
end = sp500_indices["Date"].max() + pd.Timedelta(days=5)


def fetch_fd_closes(source: str) -> pd.Series:
    toolkit = sp500_tr.to_toolkit(
        api_key=api_key,
        start_date=start.strftime("%Y-%m-%d"),
        end_date=end.strftime("%Y-%m-%d"),
        benchmark_ticker=None,
        enforce_source=source,
        progress_bar=False,
    )
    closes = toolkit.get_historical_data(
        enforce_source=source,
        show_columns=["Close"],
        include_dividends=False,
        fill_nan=False,
    )
    if closes is None or (hasattr(closes, "empty") and closes.empty):
        return pd.Series(dtype=float)
    if isinstance(closes, pd.DataFrame):
        if isinstance(closes.columns, pd.MultiIndex):
            closes = closes["Close"]
            if isinstance(closes, pd.DataFrame):
                closes = closes.iloc[:, 0]
        elif "Close" in closes.columns:
            closes = closes["Close"]
        else:
            closes = closes.squeeze()
    index = closes.index.to_timestamp() if hasattr(closes.index, "to_timestamp") else pd.to_datetime(closes.index)
    return pd.Series(closes.to_numpy(), index=index, name="Close")


fd_source = "FinancialModelingPrep"
fd_close = fetch_fd_closes(fd_source)
if fd_close.empty:
    fd_source = "YahooFinance"
    fd_close = fetch_fd_closes(fd_source)
    print(
        "FMP returned no history for ^SP500TR (still plan-gated for this ticker); "
        f"using {fd_source} via FinanceToolkit instead."
    )
else:
    print(f"FinanceDatabase prices loaded from {fd_source}.")

sp500_indices["S&P 500 TR (FinanceDatabase)"] = sp500_indices["Date"].map(fd_close)
sp500_indices

2026-10-22 09:18:57 - financetoolkit - INFO - Obtaining treasury data for 1 ticker(s)
2026-10-22 09:18:57 - financetoolkit - INFO - Obtaining historical data for 1 ticker(s)


,Period,Date,Time,S&P 500 TR (Lipper),S&P 500 TR (Yahoo),S&P 500 TR (FinanceDatabase)
0,Jan 2010,2009-12-31,Close,1837.5,1837.5,1837.5
1,Feb 2010,2010-02-01,Close,1796.67,1796.6700439453125,1796.67
2,Mar 2010,2010-03-01,Close,1844.87,1844.8699951171875,1844.87
3,Apr 2010,2010-04-01,Close,1950.91,1950.9100341796875,1950.91
4,May 2010,2010-05-03,Close,1992.87,1992.8699951171875,1992.87
...,...,...,...,...,...,...
197,Jun 2026,2026-06-01,Close,16981.1,16981.099609375,16981.0996
198,Jul 2026,2026-07-01,Close,16738.56,16738.560546875,16738.5605
199,Aug 2026,2026-08-03,Close,17012.38,17012.380859375,17012.3809
200,Sep 2026,2026-09-01,Close,17099.0,17099.0,17099.0


In [ ]:
from pathlib import Path

# Persist the aligned S&P 500 TR comparison frame for later notebooks.
CACHE_NAME = "cache-sp500-tr-indices.json"


def find_cache_path() -> Path:
    """Resolve code/cache whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        candidate = directory / "cache"
        if candidate.is_dir():
            return (candidate / CACHE_NAME).resolve()
    raise FileNotFoundError(f"Could not find cache/, starting from {start}")


cache_path = find_cache_path()
cache_path.parent.mkdir(parents=True, exist_ok=True)

cache_frame = sp500_indices.copy()
cache_frame["Date"] = cache_frame["Date"].dt.strftime("%Y-%m-%d")
cache_path.write_text(cache_frame.to_json(orient="records", indent=2) + "\n")
print(f"wrote {len(cache_frame)} rows to {cache_path}")